In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import os
from scipy.interpolate import interp1d
from scipy.optimize import brentq

plt.rcParams.update({
    "font.size": 14,                 # global font size
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "figure.figsize": (5, 4),
})


In [ ]:
# ==========================================
# 1. Helper functions
# ==========================================


def discover_temperature_dirs(base_dir):
    """Scan T_* subdirs, parse temperature, and return sorted entries."""
    raw_entries = []
    for dir_name in os.listdir(base_dir):
        if not dir_name.startswith("T_"):
            continue

        data_dir = os.path.join(base_dir, dir_name)
        if not os.path.isdir(data_dir):
            continue

        temp_str = dir_name[2:]
        try:
            T = float(temp_str)
        except ValueError:
            print(f"Skip non-numeric temperature folder: {dir_name}")
            continue

        raw_entries.append((T, dir_name, data_dir))

    if len(raw_entries) == 0:
        raise RuntimeError(
            f"No valid temperature directories found under {base_dir}. "
            "Expected names like T_0.003 or T_0.0003."
        )

    # If duplicate temperatures exist, keep lexicographically smallest folder name.
    temp_map = {}
    for T, dir_name, data_dir in raw_entries:
        if T in temp_map:
            old_name, _ = temp_map[T]
            if dir_name < old_name:
                print(f"Warning: duplicate T={T:g}, use {dir_name}, skip {old_name}")
                temp_map[T] = (dir_name, data_dir)
            else:
                print(f"Warning: duplicate T={T:g}, use {old_name}, skip {dir_name}")
        else:
            temp_map[T] = (dir_name, data_dir)

    temp_entries = [(T, info[0], info[1]) for T, info in temp_map.items()]
    temp_entries.sort(key=lambda x: x[0])
    return temp_entries


def read_opt_cond(data_dir, filename="spectra_opt_cond.csv"):
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        print(f"Warning: File not found {file_path}")
        return None

    df = pd.read_csv(file_path)
    if df.empty:
        print(f"Warning: Empty file {file_path}")
        return None

    required_cols = {"omega", "Re_Sigma"}
    if not required_cols.issubset(df.columns):
        print(f"Warning: Missing required columns in {file_path}: {required_cols - set(df.columns)}")
        return None

    df = df.sort_values("omega")
    return df

def load_summary(base_dir="./"):
    summary_path = os.path.join(base_dir, "summary_all.csv")
    if not os.path.exists(summary_path):
        return pd.DataFrame()
    return pd.read_csv(summary_path).sort_values('T')

def estimate_tc_from_summary(summary_df, rho_col='Superfluid_Stiffness_mean'):
    if summary_df.empty or rho_col not in summary_df.columns or len(summary_df) < 2:
        return np.nan
    T = summary_df['T'].to_numpy(dtype=float)
    diff = summary_df[rho_col].to_numpy(dtype=float) - (2.0 / np.pi) * T
    finite = np.isfinite(T) & np.isfinite(diff)
    T = T[finite]
    diff = diff[finite]
    if len(T) < 2:
        return np.nan
    for i in range(len(T) - 1):
        if diff[i] == 0:
            return float(T[i])
        if diff[i] * diff[i + 1] <= 0:
            return float(T[i] - diff[i] * (T[i + 1] - T[i]) / (diff[i + 1] - diff[i]))
    return np.nan


In [ ]:
# ==========================================
# 2. Main loop
# ==========================================

base_dir = "./"  # set to your data root if needed
temp_entries = discover_temperature_dirs(base_dir)
T_list = np.array([T for T, _, _ in temp_entries], dtype=float)

spectra = []

print(f"Starting optical-conductivity analysis for {len(T_list)} temperatures...")

for T, dir_name, data_dir in temp_entries:
    print(f"Processing {dir_name}...", end="\r")

    df = read_opt_cond(data_dir)
    if df is None:
        continue

    omega = df["omega"].to_numpy()
    sigma = df["Re_Sigma"].to_numpy()
    sigma_err = df["Error"].to_numpy() if "Error" in df.columns else None

    spectra.append({
        "T": T,
        "dir_name": dir_name,
        "omega": omega,
        "sigma": sigma,
        "sigma_err": sigma_err,
    })

print("\nAnalysis complete.")


In [ ]:
# ==========================================
# 3. Re sigma(omega) vs omega for different T
# ==========================================

if len(spectra) == 0:
    raise RuntimeError("No valid optical-conductivity data found.")

T_used = np.array([spec["T"] for spec in spectra], dtype=float)
# cmap = plt.cm.plasma
cmap = plt.cm.gist_rainbow_r

if len(T_used) > 1 and np.max(T_used) > np.min(T_used):
    norm = mcolors.Normalize(vmin=np.min(T_used), vmax=np.max(T_used))
else:
    t0 = float(T_used[0])
    norm = mcolors.Normalize(vmin=t0 - 1e-12, vmax=t0 + 1e-12)

fig, ax = plt.subplots(dpi=300)

for spec in spectra:
    color = cmap(norm(spec["T"]))
    ax.plot(spec["omega"], spec["sigma"], color=color, linewidth=1)

    # err = spec["sigma_err"]
    # if err is not None and len(err) == len(spec["sigma"]):
    #     mask = np.isfinite(spec["sigma"]) & np.isfinite(err)
    #     if np.any(mask):
    #         ax.fill_between(
    #             spec["omega"][mask],
    #             spec["sigma"][mask] - err[mask],
    #             spec["sigma"][mask] + err[mask],
    #             color=color,
    #             alpha=0.2,
    #         )

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r"$T$")

ax.set_xlabel(r"$\omega$")
ax.set_ylabel(r"Re $\sigma(\omega)$")
# ax.set_xlim(0,0.2)
# ax.set_ylim(2.1,3.4)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 4. Resistivity proxy R(omega)=1/Re sigma(omega) for different T
# ==========================================

if len(spectra) == 0:
    raise RuntimeError("No valid optical-conductivity data found.")

T_used = np.array([spec["T"] for spec in spectra], dtype=float)
# cmap = plt.cm.plasma
cmap = plt.cm.gist_rainbow_r

if len(T_used) > 1 and np.max(T_used) > np.min(T_used):
    norm = mcolors.Normalize(vmin=np.min(T_used), vmax=np.max(T_used))
else:
    t0 = float(T_used[0])
    norm = mcolors.Normalize(vmin=t0 - 1e-12, vmax=t0 + 1e-12)

fig, ax = plt.subplots(dpi=300)

for spec in spectra:
    omega = spec["omega"]
    sigma = spec["sigma"]
    color = cmap(norm(spec["T"]))

    mask = np.isfinite(omega) & np.isfinite(sigma) & (np.abs(sigma) > 1e-12)
    if not np.any(mask):
        continue

    R = np.full_like(sigma, np.nan, dtype=float)
    R[mask] = 1.0 / sigma[mask]
    ax.plot(omega[mask], R[mask], color=color, linewidth=1)

    # err = spec["sigma_err"]
    # if err is not None and len(err) == len(sigma):
    #     err = np.asarray(err)
    #     mask_err = mask & np.isfinite(err)
    #     if np.any(mask_err):
    #         R_err = np.full_like(sigma, np.nan, dtype=float)
    #         R_err[mask_err] = np.abs(err[mask_err] / (sigma[mask_err] ** 2))
    #         ax.fill_between(
    #             omega[mask_err],
    #             R[mask_err] - R_err[mask_err],
    #             R[mask_err] + R_err[mask_err],
    #             color=color,
    #             alpha=0.2,
    #         )

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r"$T$")

ax.set_xlabel(r"$\omega$")
ax.set_ylabel(r"$R(\omega)=1/\mathrm{Re}\,\sigma(\omega)$")
# ax.set_xlim(0,0.2)
# ax.set_ylim(0.28,0.48)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# 5. DC extrapolation (Method 1 only) + auto omega_max selection
# ==========================================

omega_fit_max_manual = 0.03
auto_select_omega_fit_max = True
omega_fit_candidates = np.round(np.arange(0.02, 0.101, 0.005), 3)

min_fit_points = 8
use_weighted_fit = False  # True uses Error column as 1/error weights

# Auto-selection hyperparameters
min_valid_ratio = 0.95
stability_tol = 2.5e-4   # median relative drift of sigma0 between neighboring windows
drift_weight = 2.0       # score = median_rel_rmse + drift_weight * median_rel_drift_prev


def extrapolate_w2_to_zero(omega, y, yerr=None, omega_max=0.03, min_points=8, use_weights=False):
    """Fit y(omega)=y0+a*omega^2 on low-frequency points and return y0 and fit diagnostics."""
    omega = np.asarray(omega, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(omega) & np.isfinite(y) & (omega >= 0.0) & (omega <= omega_max)

    weights = None
    if use_weights and (yerr is not None):
        yerr = np.asarray(yerr, dtype=float)
        mask = mask & np.isfinite(yerr) & (yerr > 0.0)

    if np.count_nonzero(mask) < min_points:
        return None

    omega_fit = omega[mask]
    y_fit = y[mask]
    x_fit = omega_fit ** 2

    if use_weights and (yerr is not None):
        weights = 1.0 / np.maximum(yerr[mask], 1e-12)

    coeff = np.polyfit(x_fit, y_fit, deg=1, w=weights)
    slope = float(coeff[0])
    y0 = float(coeff[1])

    y_pred = slope * x_fit + y0
    rmse = float(np.sqrt(np.mean((y_fit - y_pred) ** 2)))
    rel_rmse = rmse / max(abs(y0), 1e-12)

    omega_line = np.linspace(0.0, omega_max, 200)
    y_line = slope * (omega_line ** 2) + y0

    return {
        "y0": y0,
        "slope": slope,
        "n_points": int(len(omega_fit)),
        "rmse": rmse,
        "rel_rmse": rel_rmse,
        "omega_fit": omega_fit,
        "y_fit": y_fit,
        "omega_line": omega_line,
        "y_line": y_line,
    }


def scan_omega_fit_max(spectra, omega_candidates, min_points, use_weights, min_valid_ratio, stability_tol, drift_weight):
    """Scan candidate omega_max and choose a stable/accurate window globally over all temperatures."""
    rows = []
    sigma0_map = {}

    for omega_max in omega_candidates:
        sigma0_this = {}
        rel_rmse_list = []
        npt_list = []

        for spec in spectra:
            T = float(spec["T"])
            omega = np.asarray(spec["omega"], dtype=float)
            sigma = np.asarray(spec["sigma"], dtype=float)
            sigma_err = spec.get("sigma_err", None)

            fit_sigma = extrapolate_w2_to_zero(
                omega,
                sigma,
                yerr=sigma_err,
                omega_max=float(omega_max),
                min_points=min_points,
                use_weights=use_weights,
            )

            if fit_sigma is None:
                continue

            sigma0 = fit_sigma["y0"]
            if (not np.isfinite(sigma0)) or (sigma0 <= 0.0):
                continue

            sigma0_this[T] = sigma0
            rel_rmse_list.append(fit_sigma["rel_rmse"])
            npt_list.append(fit_sigma["n_points"])

        sigma0_map[float(omega_max)] = sigma0_this
        valid_count = len(sigma0_this)

        rows.append(
            {
                "omega_max": float(omega_max),
                "valid_count": valid_count,
                "valid_ratio": valid_count / max(len(spectra), 1),
                "median_rel_rmse": float(np.median(rel_rmse_list)) if len(rel_rmse_list) > 0 else np.nan,
                "p90_rel_rmse": float(np.percentile(rel_rmse_list, 90)) if len(rel_rmse_list) > 0 else np.nan,
                "min_fit_points_used": int(np.min(npt_list)) if len(npt_list) > 0 else np.nan,
                "median_fit_points_used": float(np.median(npt_list)) if len(npt_list) > 0 else np.nan,
            }
        )

    diag_df = pd.DataFrame(rows)

    drift_vals = []
    omega_list = diag_df["omega_max"].to_numpy(dtype=float)
    for i, omega_max in enumerate(omega_list):
        if i == 0:
            drift_vals.append(np.nan)
            continue

        omega_prev = omega_list[i - 1]
        map_now = sigma0_map[float(omega_max)]
        map_prev = sigma0_map[float(omega_prev)]

        common_T = sorted(set(map_now.keys()).intersection(set(map_prev.keys())))
        if len(common_T) == 0:
            drift_vals.append(np.nan)
            continue

        rel_drift = [
            abs(map_now[T] - map_prev[T]) / max(abs(map_prev[T]), 1e-12)
            for T in common_T
        ]
        drift_vals.append(float(np.median(rel_drift)))

    diag_df["median_rel_drift_prev"] = drift_vals

    drift_fill = diag_df["median_rel_drift_prev"].copy()
    if np.all(~np.isfinite(drift_fill)):
        drift_fill = pd.Series(np.zeros(len(diag_df)), index=diag_df.index, dtype=float)
    else:
        drift_fill = drift_fill.fillna(float(np.nanmax(drift_fill.to_numpy(dtype=float))))

    diag_df["score"] = diag_df["median_rel_rmse"] + drift_weight * drift_fill

    eligible = (
        (diag_df["valid_ratio"] >= min_valid_ratio)
        & np.isfinite(diag_df["score"])
        & (
            (diag_df["median_rel_drift_prev"] <= stability_tol)
            | (~np.isfinite(diag_df["median_rel_drift_prev"]))
        )
    )

    if np.any(eligible):
        best_idx = diag_df.loc[eligible, "score"].idxmin()
    else:
        valid_only = np.isfinite(diag_df["score"])
        if np.any(valid_only):
            best_idx = diag_df.loc[valid_only, "score"].idxmin()
        else:
            return float(omega_fit_max_manual), diag_df

    return float(diag_df.loc[best_idx, "omega_max"]), diag_df


if len(spectra) == 0:
    raise RuntimeError("No valid optical-conductivity data found.")

if auto_select_omega_fit_max:
    omega_fit_max, omega_scan_df = scan_omega_fit_max(
        spectra=spectra,
        omega_candidates=omega_fit_candidates,
        min_points=min_fit_points,
        use_weights=use_weighted_fit,
        min_valid_ratio=min_valid_ratio,
        stability_tol=stability_tol,
        drift_weight=drift_weight,
    )
else:
    omega_fit_max = float(omega_fit_max_manual)
    omega_scan_df = pd.DataFrame()

print(f"Selected omega_fit_max = {omega_fit_max:.3f} (auto={auto_select_omega_fit_max})")

# Method 1 only: fit sigma(omega) -> sigma_dc, then R_dc = 1/sigma_dc

dc_rows = []
fit_store = {}

for spec in spectra:
    T = float(spec["T"])
    omega = np.asarray(spec["omega"], dtype=float)
    sigma = np.asarray(spec["sigma"], dtype=float)
    sigma_err = spec.get("sigma_err", None)

    fit_sigma = extrapolate_w2_to_zero(
        omega,
        sigma,
        yerr=sigma_err,
        omega_max=omega_fit_max,
        min_points=min_fit_points,
        use_weights=use_weighted_fit,
    )

    if fit_sigma is None:
        print(f"Skip T={T:g}: not enough points for sigma-fit in [0, {omega_fit_max:.3f}]")
        continue

    sigma_dc = fit_sigma["y0"]
    if (not np.isfinite(sigma_dc)) or (sigma_dc <= 0.0):
        print(f"Skip T={T:g}: invalid sigma_dc={sigma_dc}")
        continue

    R_dc = 1.0 / sigma_dc

    dc_rows.append(
        {
            "T": T,
            "sigma_dc": sigma_dc,
            "R_dc": R_dc,
            "sigma_fit_rmse": fit_sigma["rmse"],
            "sigma_fit_rel_rmse": fit_sigma["rel_rmse"],
            "n_fit_points": fit_sigma["n_points"],
        }
    )

    fit_store[T] = {"fit_sigma": fit_sigma}

dc_df = pd.DataFrame(dc_rows).sort_values("T").reset_index(drop=True)
if len(dc_df) == 0:
    raise RuntimeError("No valid dc extrapolation result.")

print(f"Built dc extrapolation for {len(dc_df)} temperatures.")
print("Median sigma-fit relative RMSE:", float(np.median(dc_df["sigma_fit_rel_rmse"])))
dc_df.head()


In [ ]:
# ==========================================
# 6. Low-frequency sigma-fit check (sample temperatures)
# ==========================================

if len(dc_df) == 0:
    raise RuntimeError("No valid dc extrapolation result.")

n_show = 6
pick_idx = np.unique(np.linspace(0, len(dc_df) - 1, n_show, dtype=int))
T_show = dc_df.loc[pick_idx, "T"].to_numpy(dtype=float)

cmap_fit = plt.cm.gist_rainbow_r
if len(dc_df) > 1 and dc_df["T"].max() > dc_df["T"].min():
    norm_fit = mcolors.Normalize(vmin=dc_df["T"].min(), vmax=dc_df["T"].max())
else:
    t0 = float(dc_df["T"].iloc[0])
    norm_fit = mcolors.Normalize(vmin=t0 - 1e-12, vmax=t0 + 1e-12)

fig, ax = plt.subplots(figsize=(5.5, 4), dpi=300)

for T in T_show:
    color = cmap_fit(norm_fit(T))
    fit_sigma = fit_store[float(T)]["fit_sigma"]

    ax.scatter(fit_sigma["omega_fit"], fit_sigma["y_fit"], s=12, color=color, alpha=0.7)
    ax.plot(fit_sigma["omega_line"], fit_sigma["y_line"], color=color, linewidth=1.8, label=f"T={T:g}")

ax.set_title(r"Fit $\sigma(\omega)=\sigma_0 + a\omega^2$")
ax.set_xlim(0.0, omega_fit_max * 1.05)
ax.set_xlabel(r"$\omega$")
ax.set_ylabel(r"Re $\sigma(\omega)$")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(frameon=False, fontsize=10, ncol=2)

plt.tight_layout()
plt.show()


In [ ]:
summary_df = load_summary(base_dir)
Tc = estimate_tc_from_summary(summary_df)
Tc


In [ ]:
# ==========================================
# 7. DC conductivity and resistivity from summary_all.csv
# ==========================================

if 'summary_df' not in globals() or summary_df.empty:
    summary_df = load_summary(base_dir)

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.6), dpi=300, sharex=True)

if summary_df.empty or 'DC_Conductivity_mean' not in summary_df.columns:
    for ax in axes:
        ax.text(0.5, 0.5, 'No DC_Conductivity_mean in summary_all.csv',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
else:
    T = summary_df['T'].to_numpy(dtype=float)
    sigma = summary_df['DC_Conductivity_mean'].to_numpy(dtype=float)
    sigma_err = (summary_df['DC_Conductivity_err'].to_numpy(dtype=float)
                 if 'DC_Conductivity_err' in summary_df.columns else None)
    mask = np.isfinite(T) & np.isfinite(sigma) & (sigma > 0)

    axes[0].errorbar(T[mask], sigma[mask],
                     yerr=sigma_err[mask] if sigma_err is not None else None,
                     fmt='-o', capsize=3, color='tab:blue', label=r'Kubo $\sigma_{dc}$')

    R = np.full_like(sigma, np.nan, dtype=float)
    R[mask] = 1.0 / sigma[mask]
    R_err = None
    if sigma_err is not None:
        R_err = np.full_like(sigma, np.nan, dtype=float)
        R_err[mask] = sigma_err[mask] / sigma[mask]**2
    axes[1].errorbar(T[mask], R[mask],
                     yerr=R_err[mask] if R_err is not None else None,
                     fmt='-o', capsize=3, color='tab:green', label=r'$1/\sigma_{dc}$')

    if 'dc_df' in globals() and len(dc_df) > 0:
        axes[1].plot(dc_df['T'], dc_df['R_dc'], '--s', color='tab:orange',
                     label=r'optical $\omega\to0$ fit')

    for ax in axes:
        if np.isfinite(Tc):
            ax.axvline(x=Tc, color='gray', linestyle=':', linewidth=1.5,
                       label=rf'$T_c={Tc:.4f}$')
        ax.set_xlabel(r'$T$')
        ax.set_ylim(bottom=0)
        ax.legend(loc='best', frameon=False)

    axes[0].set_ylabel(r'$\sigma_{dc}$')
    axes[1].set_ylabel(r'$R_{dc}$')

plt.tight_layout()
plt.show()


In [ ]:
if len(omega_scan_df) > 0:
    fig, ax1 = plt.subplots(dpi=300)

    ax1.plot(
        omega_scan_df["omega_max"],
        100.0 * omega_scan_df["median_rel_rmse"],
        marker="o",
        linewidth=1.5,
        color="tab:blue",
        label="median rel RMSE [%]",
    )
    ax1.set_xlabel(r"$\omega_{\max}$")
    ax1.set_ylabel("median rel RMSE [%]", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax1.grid(True, linestyle="--", alpha=0.5)

    ax2 = ax1.twinx()
    ax2.plot(
        omega_scan_df["omega_max"],
        100.0 * omega_scan_df["median_rel_drift_prev"],
        marker="s",
        linewidth=1.5,
        color="tab:orange",
        label="median rel drift vs prev [%]",
    )
    ax2.set_ylabel("median rel drift vs prev [%]", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")

    ax1.axvline(omega_fit_max, color="black", linestyle="--", linewidth=1.2)
    ax1.text(
        omega_fit_max,
        ax1.get_ylim()[1] * 0.9,
        f"selected={omega_fit_max:.3f}",
        ha="left",
        va="top",
        fontsize=10,
    )

    plt.tight_layout()
    plt.show()

    display_cols = [
        "omega_max",
        "valid_count",
        "valid_ratio",
        "median_fit_points_used",
        "median_rel_rmse",
        "median_rel_drift_prev",
        "score",
    ]
    omega_scan_df[display_cols]


dc_df[["T", "sigma_dc", "R_dc", "sigma_fit_rel_rmse", "n_fit_points"]].head(10)
